# 05 — Walk-forward analysis (Phase 8)

Each fold fits on the past, selects on a validation slice that is still before the test window, then scores a **purged** out-of-sample block. Time series are never shuffled.

A mean OOS AUC near 0.5 is the usual result on noisy FX. That is information, not a failed install.

```bash
python -m quant_forex ingest --config configs/default.yaml
python -m quant_forex walk-forward --config configs/default.yaml
```


In [ ]:
from quant_forex.config.loader import load_config
from quant_forex.evaluation.folds import generate_walk_forward_folds
from quant_forex.evaluation.pipeline import run_walk_forward
from quant_forex.exploration.io import latest_processed_csv, load_processed_ohlcv
from quant_forex.features.engine import build_features
from quant_forex.models.target import TARGET_COL, add_next_return_target, feature_matrix_columns
from quant_forex.utils.paths import ProjectPaths

paths = ProjectPaths.find()
config = load_config(paths.resolve("configs/default.yaml"))
processed = latest_processed_csv(paths.data_processed)
ohlcv = load_processed_ohlcv(processed)
usable = add_next_return_target(build_features(ohlcv, config.features).frame, config.models)
cols = feature_matrix_columns(usable)
assert TARGET_COL not in cols
complete = usable.dropna(subset=cols + [TARGET_COL])
folds = generate_walk_forward_folds(len(complete), config.walk_forward, config.models.horizon)
print(config.walk_forward.mode, "folds", len(folds), "rows", len(complete))
if folds:
    print(folds[0])

In [ ]:
result = run_walk_forward(config, paths, input_csv=processed)
print(result.n_folds, result.mode)
print(result.report_path)
print(result.predictions_path)
print(result.oos_backtest)